# 02 · Cómo funciona Spark por dentro

En el primer notebook escribiste transformaciones y acciones sin preguntarte qué hacía Spark con ellas. Aquí abres el capó. El hilo es una pregunta que se repite: ¿por qué unas operaciones se paralelizan sin esfuerzo y otras obligan a mover datos entre procesos? Para responderla verás la arquitectura de un clúster de Spark (driver, ejecutores, tareas y particiones), compararás Spark con pandas y con Snowflake respondiendo la misma pregunta sobre el mismo archivo, y mirarás el plan de ejecución con `explain()` y la Spark UI para ver qué es el *shuffle*, por qué `GROUP BY` se hace en dos fases, qué cambia con un *broadcast* y por qué una ventana sin partición y un `collect` son los dos lugares donde Spark se queda trabajando como una sola máquina.

**Cómo se usa.** Ejecuta las celdas en orden, una por una, con Shift+Enter. Cada celda de código imprime lo que mide. Las cifras que dependen de la máquina las produce cada celda en tu propio entorno. Cuando el texto da una cifra, dice que es «en nuestra medición» y se refiere al Code Editor de una máquina con 2 procesadores virtuales y 7.6 GB de memoria; la tuya puede ser distinta. Hay dos problemas más de escala, los archivos pequeños y las claves desbalanceadas, que se miden en la clase 7, cuando dimensionas el cómputo. Cada celda de código empieza con un comentario `# Celda N`: es su número dentro de este notebook, y el texto lo usa cuando te manda a una celda concreta.

> **Antes de empezar.** En esta máquina solo cabe un proceso de Spark a la vez. Si ya abriste otro notebook de la clase y dejaste Spark encendido, ciérralo primero (cada notebook termina con una celda que apaga Spark) o reinicia el kernel del otro notebook.

## 0. Preparación: el entorno correcto, tu bucket y Spark encendido

Es la misma preparación del primer notebook: comprobar el kernel, armar el nombre de tu bucket, traer el archivo de enero al disco local y arrancar Spark. Si ya la ejecutaste en otro notebook, recuerda que cada notebook arranca su propio Spark y que solo puede haber uno encendido.

### 0.1 Verificar el kernel y Java

El kernel es el programa Python que ejecuta las celdas. Si eliges uno equivocado, las librerías no estarán instaladas. Elige siempre **Python (bigdata)**, que es el entorno `.venv` del proyecto. No elijas los kernels que se llaman *Glue PySpark*, *Glue Spark* o *SparkMagic*: abren una sesión de AWS Glue en la nube y esa sesión cobra mientras está abierta. La Celda 1 imprime qué Python usas y si Java está disponible, porque Spark corre sobre la máquina virtual de Java.

In [ ]:
# Celda 1
# Java: el kernel del notebook no lee ~/.bashrc, así que puede no ver el Java que instaló make setup
# en ~/.local/opt/java17. Si no está en el PATH, lo buscamos ahí y definimos JAVA_HOME antes de usar Spark.
import os, shutil
from pathlib import Path
if shutil.which("java") is None:
    for candidato in [os.environ.get("JAVA_HOME"), str(Path.home() / ".local/opt/java17")]:
        if candidato and (Path(candidato) / "bin/java").exists():
            os.environ["JAVA_HOME"] = candidato
            os.environ["PATH"] = f"{candidato}/bin:" + os.environ["PATH"]
            break

import json, re, subprocess, sys, time, contextlib, gc
from io import StringIO

print("Python:", sys.version.split()[0])
print("Ejecutable:", sys.executable)
if ".venv" not in sys.executable:
    print("AVISO: este kernel no es el del proyecto (la ruta no contiene .venv). "
          "Elige el kernel «Python (bigdata)» arriba a la derecha y reinicia el notebook.")

try:
    java = subprocess.run(["java", "-version"], capture_output=True, text=True)
    print("Java:", (java.stderr or java.stdout).splitlines()[0])
except FileNotFoundError:
    print("AVISO: no se encontró Java en el PATH; Spark no podrá arrancar.")

import pandas, pyarrow, pyspark
print("pandas", pandas.__version__, "| pyarrow", pyarrow.__version__, "| pyspark", pyspark.__version__)

### 0.2 Tu bucket, armado desde tu cuenta de AWS

Cada estudiante tiene su propio lake en S3. Su nombre no se escribe a mano: se arma con un patrón fijo, `<prefijo>-lake-<número de cuenta>-<región>`, igual que hace la función `lake_bucket` de `scripts/lib.sh` cuando corres un `make`. El número de cuenta se lo preguntamos a AWS con las credenciales del Code Editor (es lo mismo que `aws sts get-caller-identity`). La Celda 2 lo arma, lo imprime y comprueba que existe y que puedes leerlo.

In [ ]:
# Celda 2
import boto3
from botocore.exceptions import ClientError

REGION = "us-east-2"

# Raíz del repositorio: la carpeta que contiene pyproject.toml, subiendo desde donde está este notebook.
RAIZ = Path.home() / "bigdata"
for carpeta in [Path.cwd(), *Path.cwd().parents]:
    if (carpeta / "pyproject.toml").exists():
        RAIZ = carpeta
        break
sys.path.insert(0, str(RAIZ / "src"))   # así se puede hacer «from pipeline import ...»

# El prefijo viene del archivo .env del repo (lo escribe «make setup»); si no existe, es «bigdata».
prefijo = os.environ.get("PREFIX")
if not prefijo and (RAIZ / ".env").exists():
    m = re.search(r"^PREFIX=(.+)$", (RAIZ / ".env").read_text(), re.M)
    if m:
        prefijo = m.group(1).strip()
if not prefijo:
    prefijo = "bigdata"

cuenta = boto3.client("sts", region_name=REGION).get_caller_identity()["Account"]
BUCKET = f"{prefijo}-lake-{cuenta}-{REGION}"
print("Cuenta AWS :", cuenta)
print("Tu bucket  :", BUCKET)

s3 = boto3.client("s3", region_name=REGION)
try:
    s3.head_bucket(Bucket=BUCKET)
    print("Acceso al bucket: correcto")
except ClientError as e:
    print("AVISO: no se pudo leer el bucket:", e.response["Error"]["Code"],
          "- se usará el bucket público del curso para el archivo de enero.")

### 0.3 Los datos de enero, una sola vez, en el disco local

Spark lee archivos del disco de esta máquina, así que primero hay que traer el archivo de viajes de enero desde tu bucket (`bronze/_crudo/fhvhv/mes=2024-01/`, y si tu lake todavía tiene el diseño anterior, `bronze/fhvhv/mes=2024-01/`). Se guarda fuera del repositorio, en la carpeta `~/datos_clase05`, para que nunca llegue a un commit por accidente. Si la carpeta ya tiene el archivo, la celda no vuelve a descargarlo. Si tu bucket no lo tiene, usa la copia del bucket público del curso. Con las zonas pasa lo mismo, y si tampoco están en tu bucket se bajan del sitio de la TLC, que es de donde las toma `bronze.py`. La Celda 3 hace esa descarga.

In [ ]:
# Celda 3
from botocore import UNSIGNED
from botocore.config import Config
import urllib.request

DATOS = Path(os.environ.get("CLASE05_DATOS", Path.home() / "datos_clase05"))
RUTA_DIR_ENERO = DATOS / "bronze" / "fhvhv" / "mes=2024-01"
RUTA_ENERO = RUTA_DIR_ENERO / "fhvhv_tripdata_2024-01.parquet"
RUTA_ZONAS = DATOS / "bronze" / "taxi_zones" / "taxi_zone_lookup.csv"
MB = 1024 * 1024

from pipeline import bronze as bronze_pipeline   # trae URL_ZONAS, url_clima y las funciones de ingesta
from pipeline import io as pio                   # trae las rutas del lake: crudo, lotes, etc.

# El archivo crudo vive en bronze/_crudo/... (diseño nuevo). Si tu lake todavía tiene el diseño anterior
# (bronze/<fuente>/mes=.../<archivo>), la descarga prueba también esa ruta.
NOMBRE_ENERO = pio.nombre_viajes("2024-01")
CLAVES_ENERO = [pio.ruta_crudo("bronze", pio.VIAJES, NOMBRE_ENERO, "2024-01"),
                pio.ruta_anterior("bronze", pio.VIAJES, NOMBRE_ENERO, "2024-01")]
CLAVES_ZONAS = [pio.ruta_crudo("bronze", pio.ZONAS, pio.NOMBRE_ZONAS),
                pio.ruta_anterior("bronze", pio.ZONAS, pio.NOMBRE_ZONAS)]


def traer(claves, destino, url_respaldo=None):
    """Descarga a destino el primer objeto que exista entre `claves`, si destino todavía no existe.

    Busca primero en tu bucket, luego en el bucket público del curso y, si se indica, en una URL de respaldo.
    """
    if destino.exists():
        print(f"Ya existe, no se descarga: {destino}  ({destino.stat().st_size / MB:,.1f} MiB)")
        return
    destino.parent.mkdir(parents=True, exist_ok=True)
    parcial = destino.with_name(destino.name + ".parcial")   # evita dejar un archivo a medias
    t0 = time.perf_counter()
    origen = None
    for clave in claves:
        try:
            s3.download_file(BUCKET, clave, str(parcial))
            origen = f"s3://{BUCKET}/{clave}"
            break
        except ClientError:
            pass
    if origen is None and url_respaldo:
        urllib.request.urlretrieve(url_respaldo, parcial)
        origen = url_respaldo
    if origen is None:
        publico = boto3.client("s3", region_name=REGION, config=Config(signature_version=UNSIGNED))
        for clave in claves:
            try:
                publico.download_file("bigdata-udem-2026-publico", "paquete/" + clave, str(parcial))
                origen = f"s3://bigdata-udem-2026-publico/paquete/{clave}"
                break
            except ClientError:
                pass
    if origen is None:
        raise FileNotFoundError(f"No se encontró ninguno de estos objetos en tu bucket ni en el público: {claves}")
    parcial.rename(destino)
    print(f"Descargado desde {origen}\n  -> {destino}  ({destino.stat().st_size / MB:,.1f} MiB en {time.perf_counter() - t0:.1f} s)")


traer(CLAVES_ENERO, RUTA_ENERO)
traer(CLAVES_ZONAS, RUTA_ZONAS, url_respaldo=bronze_pipeline.URL_ZONAS)

### 0.4 Dos utilidades para medir

En esta clase medirás tiempo y memoria muchas veces. Para no repetir código, la Celda 4 define un cronómetro que imprime lo que tardó un bloque y lo guarda en el diccionario `TIEMPOS`, y dos funciones que leen cuánta memoria usa el proceso de este notebook. Fíjate en que la memoria que se mide es la del proceso de Python; la máquina virtual de Java que usa Spark es otro proceso aparte y se ve por separado.

In [ ]:
# Celda 4
TIEMPOS = {}


@contextlib.contextmanager
def cronometro(etiqueta):
    """Mide un bloque, lo imprime y lo guarda en TIEMPOS."""
    t0 = time.perf_counter()
    yield
    TIEMPOS[etiqueta] = time.perf_counter() - t0
    print(f"[{etiqueta}] {TIEMPOS[etiqueta]:.2f} s")


def rss_mb():
    """Memoria residente actual del proceso del notebook (MB). En macOS no está disponible."""
    try:
        with open("/proc/self/status") as f:
            for linea in f:
                if linea.startswith("VmRSS:"):
                    return int(linea.split()[1]) / 1024
    except FileNotFoundError:
        pass
    return float("nan")


def pico_mb():
    """Máximo de memoria residente que ha alcanzado el proceso del notebook desde que arrancó (MB)."""
    import resource
    pico = resource.getrusage(resource.RUSAGE_SELF).ru_maxrss
    if sys.platform == "darwin":
        return pico / MB
    return pico / 1024


def fmt(n):
    """19663930 -> '19 663 930'"""
    return f"{n:,}".replace(",", " ")


def memoria_equipo_gb():
    return os.sysconf("SC_PAGE_SIZE") * os.sysconf("SC_PHYS_PAGES") / 1024**3


print(f"Memoria total de esta máquina: {memoria_equipo_gb():.1f} GB")
print(f"Memoria del proceso del notebook ahora: {rss_mb():,.0f} MB")

### 0.5 Arrancar Spark con la misma configuración del pipeline

No vamos a inventar una configuración aparte para el notebook: usamos la misma función `get_spark("local")` de `src/pipeline/engine.py` que usa el pipeline. Crea una sesión `local[*]`, que significa que todo vive en un solo proceso de Java: el driver y un único ejecutor, que tiene tantos hilos como núcleos tenga la máquina (dos en el Code Editor). No hay dos ejecutores: hay uno con dos hilos, y cada hilo corre una tarea a la vez. El driver tiene 2 GB de memoria, la zona horaria de la sesión queda en UTC y los reemplazos de particiones son dinámicos.

Después de arrancar, la Celda 5 imprime `defaultParallelism`, el número de tareas que Spark puede ejecutar al mismo tiempo, y el enlace a la Spark UI, la página donde Spark muestra cada trabajo, etapa y tarea mientras corre.

In [ ]:
# Celda 5
# La Spark UI se ve a través del proxy del Code Editor, que agrega el prefijo /codeeditor/default/ports/4040.
# Spark necesita saberlo para armar sus enlaces, estilos e imágenes con ese prefijo; si no, la página
# carga sin diseño. Se indica antes de arrancar Spark, porque se lee al encender la JVM.
import os
os.environ["PYSPARK_SUBMIT_ARGS"] = "--conf spark.ui.proxyBase=/codeeditor/default/ports/4040 pyspark-shell"

from pyspark.sql import functions as F, DataFrame, Window
from pipeline.engine import get_spark

with cronometro("arrancar Spark"):
    spark = get_spark("local")

print("Versión de Spark :", spark.version)
print("Master           :", spark.sparkContext.master)
print("Memoria del driver:", spark.sparkContext.getConf().get("spark.driver.memory"))
print("defaultParallelism (tareas simultáneas):", spark.sparkContext.defaultParallelism)


def url_spark_ui():
    """Enlace a la Spark UI. Dentro del Code Editor pasa por el proxy del Space, igual que Airflow."""
    local = spark.sparkContext.uiWebUrl            # p. ej. http://10.0.0.5:4040
    puerto = local.rsplit(":", 1)[-1].strip("/")
    meta = Path("/opt/ml/metadata/resource-metadata.json")
    if meta.exists():
        try:
            datos = json.loads(meta.read_text())
            r = boto3.client("sagemaker", region_name=REGION).describe_space(
                DomainId=datos["DomainId"], SpaceName=datos["SpaceName"])
            base = re.match(r"https://[^/]+", r["Url"]).group(0)
            return f"{base}/codeeditor/default/ports/{puerto}/jobs/"   # /jobs/ directo: la raíz redirige mal detrás del proxy
        except Exception as e:                      # sin permiso o fuera del Code Editor
            print("(no se pudo consultar el Space:", type(e).__name__, ")")
    return f"{local}   (en el Code Editor: https://<id-del-space>.studio.{REGION}.sagemaker.aws/codeeditor/default/ports/{puerto}/)"


print("Spark UI:", url_spark_ui())

**Qué observar.** `defaultParallelism` es el número de hilos del único ejecutor de esta máquina; esa cifra manda cuántas tareas avanzan a la vez. Abre el enlace de la Spark UI en otra pestaña y déjalo abierto: la pestaña *Jobs* está vacía porque Spark todavía no ha hecho nada, y se irá llenando a medida que ejecutes celdas. La UI solo existe mientras la sesión de Spark está viva.

## 1. Arquitectura: driver, ejecutores, tareas y particiones

**El problema.** Pandas carga la tabla completa en la memoria de un solo proceso. Mientras la tabla quepa, es rapidísimo; cuando no cabe, el proceso se cae. Con 19.6 millones de filas ya se nota, y con los 239.5 millones de viajes de 2024 el problema es de otra escala. **La idea de ingeniería** es no cargar todo: partir los datos en trozos independientes, las *particiones*, y repartir el trabajo entre varios procesadores. **Lo que resuelve Spark** es exactamente eso: planificar el trabajo, repartirlo y juntar los resultados. Lo que no resuelve es la lógica: si tu transformación obliga a juntar todo en un solo lugar, Spark también se queda esperando.

### Los cuatro nombres de la arquitectura

El *driver* es el proceso que lee tu código, construye el plan de ejecución y reparte el trabajo. Un *ejecutor* es un proceso que hace el trabajo sobre los datos y puede correr varias tareas a la vez, tantas como núcleos tenga. Una *partición* es un trozo de los datos que se procesa de una sola vez. Una *tarea* es el trabajo de procesar una partición en una etapa, y corre en un hilo del ejecutor. En un clúster real los ejecutores son procesos repartidos en varias máquinas. En el Code Editor, con `local[*]`, hay un solo proceso de Java que hace de driver y de ejecutor a la vez: el ejecutor es uno solo y tiene dos hilos, uno por núcleo, así que corren dos tareas al mismo tiempo (no hay dos ejecutores). La lógica es idéntica, solo cambia dónde viven el ejecutor y sus hilos.

La Celda 6 lo comprueba con tus datos: abre el archivo de enero sin cargar sus filas y cuenta cuántas particiones forma Spark. La Celda 7 cuenta cuántas filas caen en cada una.

In [ ]:
# Celda 6
with cronometro("leer metadatos del archivo (sin cargar filas)"):
    viajes_sp = spark.read.parquet(str(RUTA_DIR_ENERO))   # lee solo el esquema (el pie del archivo), no las filas

n_part = viajes_sp.rdd.getNumPartitions()
print("Particiones que Spark formó para el archivo de enero:", n_part)
print("Tamaño máximo de partición al leer archivos (bytes):", spark.conf.get("spark.sql.files.maxPartitionBytes"))
print("Tareas que pueden correr a la vez:", spark.sparkContext.defaultParallelism)

In [ ]:
# Celda 7
with cronometro("filas por partición"):
    por_particion = (viajes_sp
                     .groupBy(F.spark_partition_id().alias("particion"))
                     .count()
                     .orderBy("particion")
                     .toPandas())

print(por_particion.to_string(index=False))
print("Suma de las particiones:", fmt(int(por_particion["count"].sum())), "filas")

**Qué observar.** Cada fila de la tabla es una partición, y cada una será una tarea. Dos cosas distintas deciden el reparto, y ninguna la decidiste tú. El *número* de particiones sale de los bytes: Spark corta el archivo en trozos de hasta `spark.sql.files.maxPartitionBytes` (128 MiB) y, si la máquina tiene varios núcleos para repartir, en trozos más pequeños; en el Code Editor, con 2 núcleos, los 451 MiB de enero dan 4 particiones. Los *grupos de filas* del Parquet, de 1 048 576 filas cada uno, deciden **qué filas** caen en cada partición: un grupo no se parte, así que las filas por partición son múltiplos de 1 048 576 (en el Code Editor, 5 242 880 es 5 × 1 048 576 y 6 291 456 es 6 × 1 048 576), salvo la última, que se lleva el resto. Por eso las particiones no quedan del mismo tamaño. Compara el número de particiones con `defaultParallelism`: si hay más particiones que hilos, las tareas esperan turno en grupos; si hay menos, algunos hilos se quedan sin trabajo. La suma de las filas por partición debe dar las 19 663 930 filas del archivo (la celda las imprime para que lo compruebes). Si abres la Spark UI en la pestaña *Jobs*, verás el trabajo que acabas de lanzar y, dentro de él, una tarea por partición.

## 2. Pandas, Spark y Snowflake responden la misma pregunta

Para que la comparación sea justa, los tres motores responden **exactamente la misma pregunta** sobre **el mismo archivo** de enero: ¿cuántos viajes hubo y cuánto se facturó en tarifa base (`base_passenger_fare`) por zona de recogida (`PULocationID`) y por día de recogida? Pandas y Spark leen el archivo desde el disco local, que viene de tu bucket. Snowflake lo lee desde el bucket público del curso, que contiene el mismo archivo de enero, sin credenciales. Al final comprobarás que los tres devuelven los mismos totales; si no coincidieran, la comparación de tiempos no tendría sentido.

Los tiempos que verás dependen de esta máquina. Por eso el texto no trae cifras propias: las produce cada celda. Las cifras que sí aparecen, marcadas como «en nuestra medición», son las del Code Editor de una máquina con 2 procesadores virtuales y 7.6 GB de memoria. La Celda 8 fija la pregunta, y la Celda 9, la Celda 11 y la Celda 12 la responden con cada motor.

In [ ]:
# Celda 8
INICIO, FIN = "2024-01-01", "2024-02-01"            # enero, fin exclusivo
COLUMNAS_PREGUNTA = ["pickup_datetime", "PULocationID", "base_passenger_fare"]
RESULTADOS = {}                                      # aquí queda lo que responde cada motor

### 2.1 Pandas: leer solo lo necesario

El archivo tiene 24 columnas, pero la pregunta usa tres. Parquet guarda los datos por columna, así que se pueden leer solo esas tres sin tocar las demás. Eso lo hace `pyarrow` con el argumento `columns=`. La celda mide cada paso por separado: leer, convertir a un DataFrame de pandas y calcular. También guarda el pico de memoria del proceso, que es el dato que más importa con pandas.

In [ ]:
# Celda 9
import pyarrow.parquet as pq
import pandas as pd

rss_antes, pico_antes = rss_mb(), pico_mb()

with cronometro("pandas 1/3: leer 3 columnas con pyarrow"):
    tabla = pq.read_table(RUTA_ENERO, columns=COLUMNAS_PREGUNTA)
with cronometro("pandas 2/3: convertir a DataFrame"):
    viajes_pd = tabla.to_pandas()
mb_df = viajes_pd.memory_usage(deep=True).sum() / MB
with cronometro("pandas 3/3: filtrar, agrupar y sumar"):
    enero = viajes_pd[(viajes_pd["pickup_datetime"] >= INICIO) & (viajes_pd["pickup_datetime"] < FIN)]
    res_pandas = (enero.assign(fecha=enero["pickup_datetime"].dt.normalize())
                  .groupby(["PULocationID", "fecha"], as_index=False)
                  .agg(viajes=("base_passenger_fare", "size"), ingreso=("base_passenger_fare", "sum")))

pico_despues = pico_mb()
segundos_pandas = 0.0
for etiqueta in TIEMPOS:
    if etiqueta.startswith("pandas "):
        segundos_pandas = segundos_pandas + TIEMPOS[etiqueta]
RESULTADOS["pandas"] = dict(
    df=res_pandas.rename(columns={"PULocationID": "pu_location_id"}),
    segundos=segundos_pandas,
    pico_mb=pico_despues,
)
print(f"\nFilas leídas: {fmt(len(viajes_pd))} | filas en enero: {fmt(len(enero))} | grupos zona-día: {fmt(len(res_pandas))}")
print(f"DataFrame con 3 columnas: {mb_df:,.0f} MB")
print(f"Memoria del proceso: antes {rss_antes:,.0f} MB | pico del proceso hasta aquí {pico_despues:,.0f} MB (antes de la celda: {pico_antes:,.0f} MB)")
res_pandas.head()

**Qué observar.** El DataFrame con solo tres columnas ocupa una cantidad de memoria que ya se ve en la celda, y el *pico* del proceso es bastante mayor que eso, porque pandas hace copias intermedias al filtrar, al crear la columna `fecha` y al agrupar. El tiempo de pandas suele ser corto: cuando los datos caben en memoria, pandas es difícil de superar. La pregunta de ingeniería es qué pasa cuando ya no caben.

### 2.2 Qué pasaría con el año completo

Enero con tres columnas cabe en pandas. La pregunta de ingeniería es qué pasa con el año: 2024 trae 239.5 millones de viajes, unas 12 veces los de enero. La Celda 10 responde sin arriesgar el kernel, con aritmética y no con una medición: multiplica el pico de memoria que alcanzó pandas en la Celda 9 por la razón entre los viajes del año y los de enero. Además lee **un solo grupo de filas** con las 24 columnas, mide cuánta memoria ocupa y extrapola de la misma forma, para el caso de una pregunta que pidiera todas las columnas. Las columnas de texto (`hvfhs_license_num`, `dispatching_base_num` y las banderas) son las que más pesan en pandas, porque cada valor se convierte en un objeto de Python.

In [ ]:
# Celda 10
del tabla, viajes_pd, enero
gc.collect()

archivo = pq.ParquetFile(RUTA_ENERO)
grupo = archivo.read_row_group(0)                       # todas las columnas, un solo grupo de filas
mb_grupo = grupo.to_pandas().memory_usage(deep=True).sum() / MB
filas_archivo = archivo.metadata.num_rows
est_mes = mb_grupo * filas_archivo / grupo.num_rows
FILAS_ANIO = 239_500_000                                 # viajes FHVHV de 2024 (dato del caso del curso)
est_anio = est_mes * FILAS_ANIO / filas_archivo

# El argumento que importa: el año completo con las mismas 3 columnas de la pregunta.
# Se usa el pico de memoria de la celda de pandas, porque pandas hace copias al filtrar y agrupar.
est_anio_3_columnas = pico_despues * FILAS_ANIO / filas_archivo

print(f"Año completo, solo las 3 columnas de la pregunta: {est_anio_3_columnas / 1024:,.1f} GB (estimado con el pico de pandas)")
print(f"Un grupo de {fmt(grupo.num_rows)} filas con las {grupo.num_columns} columnas ocupa {mb_grupo:,.0f} MB en pandas (medido).")
print(f"Estimación para enero completo con las 24 columnas: {est_mes / 1024:,.1f} GB")
print(f"Estimación para el año con las 24 columnas (239.5 M de viajes): {est_anio / 1024:,.1f} GB")
print(f"Memoria total de esta máquina: {memoria_equipo_gb():.1f} GB")
del grupo
gc.collect()

**Qué observar.** Compara la primera cifra con la memoria total de la máquina. En nuestra medición en el Code Editor, el año con solo las tres columnas de la pregunta se estima en unos 32 GB, frente a 7.6 GB de memoria: pandas no puede con el año, ni siquiera pidiendo solo lo que necesita. Las cifras con las 24 columnas son todavía mayores, pero nadie pediría esas columnas para esta pregunta, así que no son el argumento. Hay tres salidas. Una es una máquina mucho más grande (escalar hacia arriba, con un costo que crece). Otra es una herramienta que procese por partes dentro de una sola máquina: DuckDB o Polars también resolverían esta pregunta sobre el año sin salir de una sola máquina. La tercera es Spark, que procesa partición por partición, escribe en disco lo que no cabe en memoria y tiene una ventaja que las otras no tienen: **el mismo código escala sin cambios a Glue**, donde los ejecutores son varias máquinas. Esa tercera razón, y no la velocidad en una sola máquina, es la que justifica Spark en este curso. Y siempre queda la defensa que ya usaste: no leer lo que no necesitas.

### 2.3 Spark: la misma pregunta con SQL y con la API de DataFrames

Spark permite escribir la consulta de dos maneras equivalentes. La primera registra el DataFrame como una vista temporal y usa SQL; la segunda encadena métodos del DataFrame. Las dos producen **el mismo plan de ejecución**, así que ninguna debería ser más rápida que la otra. La Celda 11 ejecuta primero la consulta SQL y después la de DataFrames, y la segunda encuentra la máquina virtual de Java ya caliente y el archivo en la caché de disco del sistema operativo. Si una tarda menos, es por el orden en que corrieron y no por la API.

In [ ]:
# Celda 11
viajes_sp.createOrReplaceTempView("viajes")

rss_antes = rss_mb()
with cronometro("spark SQL"):
    res_sql = spark.sql(f"""
        SELECT PULocationID AS pu_location_id,
               to_date(pickup_datetime) AS fecha,
               COUNT(*) AS viajes,
               SUM(base_passenger_fare) AS ingreso
        FROM viajes
        WHERE pickup_datetime >= '{INICIO}' AND pickup_datetime < '{FIN}'
        GROUP BY PULocationID, to_date(pickup_datetime)
    """).toPandas()

with cronometro("spark DataFrame"):
    res_df = (viajes_sp
              .where((F.col("pickup_datetime") >= INICIO) & (F.col("pickup_datetime") < FIN))
              .groupBy(F.col("PULocationID").alias("pu_location_id"),
                       F.to_date("pickup_datetime").alias("fecha"))
              .agg(F.count(F.lit(1)).alias("viajes"), F.sum("base_passenger_fare").alias("ingreso"))
              .toPandas())

RESULTADOS["spark"] = dict(df=res_sql, segundos=TIEMPOS["spark SQL"], pico_mb=pico_mb())
print(f"\nGrupos zona-día: SQL {fmt(len(res_sql))} | DataFrame {fmt(len(res_df))}")
print(f"Memoria del proceso del notebook (Python): {rss_antes:,.0f} MB antes, {rss_mb():,.0f} MB después")
print("La memoria de Spark vive en el proceso Java (driver de 2 GB), no en este.")
res_sql.head()

**Qué observar.** Primero, que el DataFrame y el SQL dan el mismo número de grupos. Segundo, que la memoria del proceso de Python casi no se movió: Spark no pasó los 19.6 millones de filas por Python, solo devolvió el resultado final (unos miles de filas) con `toPandas()`. Tercero, que Spark nunca tuvo todo el archivo en memoria a la vez: leyó una partición por tarea, la agregó y siguió. En la Spark UI, pestaña *Jobs*, verás los dos trabajos que acabas de ejecutar. Cuarto, compara los dos tiempos. En nuestra medición, la consulta SQL tardó 7.8 segundos y la de DataFrames 4.9, pero esa diferencia no viene de la API: las dos generan el mismo plan físico (en la pestaña *SQL / DataFrame* de la Spark UI los dos diagramas son iguales). Viene del orden, porque la primera paga el calentamiento de Java y del disco. Para comprobarlo, vuelve a ejecutar esta misma celda: con todo caliente, los dos tiempos se parecen.

### 2.4 Snowflake, desde el notebook

Snowflake puede leer un archivo Parquet que está en S3 sin cargarlo antes a una tabla: se crea un *stage* (una referencia a una ubicación de archivos) y se consulta con SQL. Aquí el stage es **temporal** (desaparece al cerrar la sesión) y apunta al bucket público del curso, que se puede leer sin credenciales. La conexión usa la llave que ya configuraste: `snowflake.connector.connect(connection_name="default")` lee `~/.snowflake/connections.toml`, el mismo archivo que usa `snow`.

Dos detalles de medición que importan. Uno: Snowflake guarda el resultado de cada consulta y, si repites la misma, lo devuelve sin ejecutar nada; para medir de verdad se desactiva con `USE_CACHED_RESULT = FALSE`. Dos: si el warehouse estaba suspendido, la primera consulta paga el arranque; por eso se hace antes una consulta trivial y su tiempo se mide aparte. El tiempo de la consulta incluye leer 451 MiB desde S3, interpretar el Parquet y bajar el resultado al notebook.

Para leer el Parquet se usa `USE_VECTORIZED_SCANNER = TRUE` en el formato del stage. Sin esa opción Snowflake interpreta cada fila como un valor semiestructurado (`VARIANT`) y la misma consulta puede tardar muchas veces más. Si tienes curiosidad, quita esa opción del formato, repite la celda y compara el tiempo.

In [ ]:
# Celda 12
import snowflake.connector

SF_BASE, SF_ESQUEMA = "CURSO", "CLASE5"      # una base y un esquema solo para esta comparación
BUCKET_PUBLICO = "s3://bigdata-udem-2026-publico/paquete/bronze/fhvhv/mes=2024-01/"

con = snowflake.connector.connect(connection_name="default")
cur = con.cursor()

def sf(sql):
    """Ejecuta una sentencia y devuelve el cursor."""
    return cur.execute(sql)

# Rol y warehouse explícitos: la conexión por llave no trae warehouse por defecto (CURRENT_WAREHOUSE() da None).
sf("USE ROLE ACCOUNTADMIN")       # el administrador de tu cuenta de prueba
sf("USE WAREHOUSE COMPUTE_WH")    # el warehouse X-Small que viene con toda cuenta de prueba
rol, warehouse = sf("SELECT CURRENT_ROLE(), CURRENT_WAREHOUSE()").fetchone()
print("Rol:", rol, "| Warehouse:", warehouse)

sf(f"CREATE DATABASE IF NOT EXISTS {SF_BASE}")          # si ya existe, no la toca
sf(f"CREATE SCHEMA IF NOT EXISTS {SF_BASE}.{SF_ESQUEMA}")
sf(f"USE SCHEMA {SF_BASE}.{SF_ESQUEMA}")
sf("ALTER SESSION SET USE_CACHED_RESULT = FALSE")      # medir de verdad, sin devolver resultados guardados

with cronometro("snowflake: despertar el warehouse (SELECT 1)"):
    sf("SELECT 1").fetchall()

sf(f"""
    CREATE OR REPLACE TEMPORARY STAGE viajes_enero
    URL = '{BUCKET_PUBLICO}'
    FILE_FORMAT = (TYPE = PARQUET USE_LOGICAL_TYPE = TRUE USE_VECTORIZED_SCANNER = TRUE)
""")
archivos_en_stage = []
for fila in sf("LIST @viajes_enero").fetchall():
    archivos_en_stage.append((fila[0].split("/")[-1], fila[1]))
print("Archivos en el stage:", archivos_en_stage)

CONSULTA_SF = f"""
    SELECT $1:PULocationID::INT                       AS pu_location_id,
           $1:pickup_datetime::TIMESTAMP_NTZ::DATE    AS fecha,
           COUNT(*)                                   AS viajes,
           SUM($1:base_passenger_fare::DOUBLE)        AS ingreso
    FROM @viajes_enero
    WHERE $1:pickup_datetime::TIMESTAMP_NTZ >= '{INICIO}'
      AND $1:pickup_datetime::TIMESTAMP_NTZ <  '{FIN}'
    GROUP BY 1, 2
"""
with cronometro("snowflake: consulta de la pregunta"):
    sf(CONSULTA_SF)
    res_sf = cur.fetch_pandas_all()
nombres_en_minuscula = []
for nombre in res_sf.columns:
    nombres_en_minuscula.append(nombre.lower())
res_sf.columns = nombres_en_minuscula
res_sf["pu_location_id"] = res_sf["pu_location_id"].astype("int64")
print("ID de la consulta (búscala en Snowsight > Monitoring > Query History):", cur.sfqid)
RESULTADOS["snowflake"] = dict(df=res_sf, segundos=TIEMPOS["snowflake: consulta de la pregunta"], pico_mb=pico_mb())
res_sf.head()

**Qué observar.** El tiempo de esta celda no es comparable con el de pandas o Spark «en abstracto»: incluye leer el archivo por la red, y Snowflake trabaja sobre un warehouse del tamaño que tenga tu cuenta. Lo que sí es comparable es que la consulta SQL es casi la misma que escribiste en Spark; cambia la forma de apuntar a la tabla (`@viajes_enero` y las columnas como `$1:nombre`). Si quieres ver cómo se repartió el trabajo, abre la consulta en el *Query Profile* de Snowsight con el ID que imprimió la celda: es la misma lectura de evidencia que hiciste en la clase 3.

### 2.5 Los tres resultados coinciden

Comparar tiempos solo tiene sentido si los tres respondieron lo mismo. La Celda 13 compara, motor contra motor, el número de grupos zona-día, el total de viajes, el total de ingreso (con una tolerancia mínima, porque sumar decimales en distinto orden puede cambiar la última cifra) y, grupo por grupo, cuántos viajes difieren.

In [ ]:
# Celda 13
import math

def normalizar(df):
    out = df[["pu_location_id", "fecha", "viajes", "ingreso"]].copy()
    out["pu_location_id"] = out["pu_location_id"].astype("int64")
    out["fecha"] = pd.to_datetime(out["fecha"]).dt.date
    out["viajes"] = out["viajes"].astype("int64")
    out["ingreso"] = out["ingreso"].astype("float64")
    return out.sort_values(["pu_location_id", "fecha"]).reset_index(drop=True)

filas = []
for motor, r in RESULTADOS.items():
    r["norm"] = normalizar(r["df"])
    filas.append(dict(motor=motor, grupos=len(r["norm"]), viajes=int(r["norm"]["viajes"].sum()),
                      ingreso=round(float(r["norm"]["ingreso"].sum()), 2), segundos=round(r["segundos"], 1)))
resumen = pd.DataFrame(filas).set_index("motor")
print(resumen.to_string(formatters={"ingreso": "{:,.2f}".format}))

base = RESULTADOS["pandas"]["norm"]
for motor, r in RESULTADOS.items():
    if motor == "pandas":
        continue
    cruce = base.merge(r["norm"], on=["pu_location_id", "fecha"], how="outer", suffixes=("_pd", "_x"), indicator=True)
    sin_pareja = int((cruce["_merge"] != "both").sum())
    dif_viajes = int((cruce["viajes_pd"] != cruce["viajes_x"]).sum())
    dif_ingreso = float((cruce["ingreso_pd"] - cruce["ingreso_x"]).abs().max())
    ok = (sin_pareja == 0 and dif_viajes == 0 and dif_ingreso < 1e-3
          and math.isclose(resumen.loc["pandas", "ingreso"], resumen.loc[motor, "ingreso"], rel_tol=1e-9))
    if ok:
        veredicto = "COINCIDEN"
    else:
        veredicto = "REVISAR"
    print(f"pandas vs {motor}: grupos sin pareja = {sin_pareja}, grupos con viajes distintos = {dif_viajes}, "
          f"mayor diferencia de ingreso = {dif_ingreso:.6f}  ->  {veredicto}")

**Qué observar.** La tabla pone lado a lado, para cada motor, cuántos grupos produjo, cuántos viajes sumó, el ingreso total y los segundos que tardó. Las líneas `pandas vs ...` deben decir `COINCIDEN`. Si alguna dice `REVISAR`, no sigas: significa que los motores no respondieron la misma pregunta (un filtro de fechas distinto, una zona nula, un tipo mal convertido) y los tiempos no son comparables. Cuando todo coincide, lee los tiempos con cuidado: con un mes y tres columnas, los tres motores tardan tiempos del mismo orden y ninguno gana con claridad. Pandas es difícil de superar mientras los datos caben en memoria; Spark empieza a justificar su costo de arranque cuando los datos o el cálculo ya no caben en una máquina o cuando el código tiene que correr en un clúster; Snowflake paga por leer por la red, pero no necesita que tengas nada instalado ni memoria propia. Ninguno es «el mejor»: es un compromiso entre dónde están los datos, cuánto pesan y quién paga el cómputo.

Cierra la conexión de Snowflake (la Celda 14) cuando termines de usarla (el warehouse se suspende solo, pero la conexión abierta no hace falta):

In [ ]:
# Celda 14
cur.close()
con.close()
print("Conexión a Snowflake cerrada.")

### 2.6 Cuándo conviene cada uno

Los tres motores respondieron lo mismo, y eso permite comparar lo que de verdad los distingue. En nuestra medición en el Code Editor (una máquina de 2 procesadores virtuales y 7.6 GB de memoria), pandas tardó 5.7 segundos con un pico de memoria de 2.64 GB. Spark tardó 7.8 segundos la primera vez y 4.9 la segunda: la consulta en SQL y la de DataFrames generan el mismo plan, y la diferencia se debe a que la segunda corrió con Java y el disco ya calientes. Encender Spark suma además de 7 a 10 segundos que no están en esa cifra, y el proceso de Python se queda entre 381 y 198 MB porque el trabajo pesado ocurre en Java. Snowflake con un warehouse X-Small tardó 10.9 segundos usando el lector vectorizado. Con tres columnas y un mes, entonces, los tres motores dan el mismo resultado en tiempos parecidos, y en una sola máquina Spark no gana. En tu máquina los tiempos serán otros, y lo que no cambia es la forma de cada motor:

| | pandas | Spark (local o en un clúster) | Snowflake |
|---|---|---|---|
| Dónde corre el cálculo | En un solo proceso de Python, en tu máquina | En el driver y los ejecutores; en el Code Editor, hilos de tu máquina; en Glue, varias máquinas | En el warehouse, dentro de Snowflake |
| Dónde están los datos | Tienen que caber en la memoria de ese proceso | En archivos que se leen por particiones; no hace falta que quepan en memoria | En tablas de Snowflake, o en archivos externos como el stage de esta sección |
| Qué pasa cuando los datos crecen | Se queda sin memoria: necesitas una máquina más grande | Se agregan más ejecutores, y las operaciones con *shuffle* cuestan más | Se elige un warehouse más grande o se agregan clústeres |
| Costo de arrancar | Casi cero | Segundos para encender la JVM y planificar | Segundos si el warehouse estaba suspendido |
| Lo que tienes que saber | Python y pandas | Python y el modelo de particiones, etapas y shuffle | SQL y cómo leer el *Query Profile* |
| Cuándo es la mejor opción | Los datos caben en memoria y la pregunta es corta | El proceso es una cadena de transformaciones sobre archivos grandes, y tiene que correr de forma repetible (un pipeline) | La pregunta es analítica, el dato ya vive en Snowflake y quieres cero infraestructura |

Ninguno es «el mejor»: es un compromiso entre dónde están los datos, cuánto pesan y quién paga el cómputo. En este curso el pipeline usa Spark porque corre igual en tu máquina, en Airflow local y en Glue, y los resultados llegan a Snowflake para consultarlos. Herramientas como DuckDB o Polars también procesarían un año de esta pregunta en una sola máquina; Spark se elige aquí porque el mismo código escala sin cambios a Glue, donde los ejecutores son varias máquinas.

## 3. Lazy y `explain()`: Spark anota lo que quieres hacer y solo trabaja cuando le pides un resultado

**El problema.** Si cada paso de una transformación se ejecutara al instante, Spark leería 19.6 millones de filas para filtrar, escribiría el resultado intermedio, lo volvería a leer para seleccionar columnas y así sucesivamente. **La idea de ingeniería** es separar *describir* de *ejecutar*. Las *transformaciones* (`where`, `select`, `withColumn`, `groupBy`, `join`) solo agregan un paso al plan y devuelven otro DataFrame al instante. Las *acciones* (`count`, `collect`, `show`, `toPandas`, escribir un archivo) son las que obligan a ejecutar el plan completo. **Lo que resuelve** esa pereza (*lazy evaluation*) es que Spark ve el plan entero antes de tocar un dato, y puede leer solo las columnas que de verdad se usan y empujar los filtros hasta el lector del archivo. **Lo que no resuelve**: los errores que dependen de los datos o de los archivos solo aparecen cuando ejecutas una acción, no cuando escribes la transformación. Los nombres de columna y los tipos sí se revisan al definir el paso.

La Celda 15 construye una cadena de transformaciones y mide cuánto tarda **definirla**. La Celda 16 ejecuta una acción y mide cuánto tarda **hacerlo**. Después, la Celda 17 pide el plan sin ejecutarlo.

In [ ]:
# Celda 15
with cronometro("definir: leer"):
    v = spark.read.parquet(str(RUTA_DIR_ENERO))
with cronometro("definir: where"):
    v = v.where(F.col("PULocationID") == 132)
with cronometro("definir: select"):
    v = v.select("pickup_datetime", "base_passenger_fare")
with cronometro("definir: withColumn"):
    v = v.withColumn("tarifa_centavos", F.col("base_passenger_fare") * 100)

print("\nNo se ha leído ninguna fila de datos todavía; solo se anotó el plan.")

In [ ]:
# Celda 16
with cronometro("acción: count() ejecuta todo el plan"):
    n = v.count()
print(f"Viajes que salen de la zona 132: {fmt(n)}")

**Qué observar.** Las cuatro líneas de «definir» tardan milisegundos, sin importar que el archivo pese 451 MiB: Spark apenas leyó el pie del archivo Parquet para conocer sus columnas. El tiempo real aparece en el `count()`, que es la acción. Si abres la Spark UI en *Jobs*, ahí está el trabajo de ese `count()`; las transformaciones no generaron ninguno.

### El plan con y sin filtro: lectura de columnas y filtros empujados

`explain()` imprime el plan físico, que se lee de abajo hacia arriba: la última línea es la lectura del archivo (`FileScan parquet`) y las de arriba son lo que se hace con esos datos. En esa línea buscamos dos cosas: `ReadSchema`, las únicas columnas que Spark va a leer del Parquet, y `PushedFilters`, los filtros que se le entregan al lector del archivo para que descarte datos antes de subirlos a memoria. Esa es la ganancia de ser lazy: Spark pudo mirar todo el plan y pedirle al archivo solo lo necesario.

In [ ]:
# Celda 17
def texto_plan(df, modo="simple"):
    """El plan de df como texto (explain() imprime; aquí se captura)."""
    salida = StringIO()
    with contextlib.redirect_stdout(salida):
        df.explain(mode=modo)
    return salida.getvalue()


def mostrar_plan(df, claves=None):
    """Imprime el plan; si se dan claves, solo las líneas que contienen alguna de ellas.
    La línea del FileScan es larguísima: se resume a sus filtros empujados y a las columnas que lee."""
    for linea in texto_plan(df).splitlines():
        if claves is not None:
            contiene = False
            for clave in claves:
                if clave in linea:
                    contiene = True
            if not contiene:
                continue
        if "FileScan" in linea:
            trozos = []
            for patron in (r"PushedFilters: \[.*?\]", r"ReadSchema: .*$"):
                encontrado = re.search(patron, linea)
                if encontrado:
                    trozos.append(encontrado.group(0))
            linea = "FileScan parquet | " + " | ".join(trozos)
        print(linea.strip()[:400])


base_v = spark.read.parquet(str(RUTA_DIR_ENERO))
print("=== Sin filtro: select de 2 columnas ===")
mostrar_plan(base_v.select("PULocationID", "base_passenger_fare"), ["FileScan", "Project"])
print("\n=== Con filtro: where PULocationID = 132 ===")
mostrar_plan(base_v.where(F.col("PULocationID") == 132).select("PULocationID", "base_passenger_fare"),
             ["FileScan", "Filter", "Project"])

**Qué observar.** En las dos salidas, `ReadSchema` lista solo `PULocationID` y `base_passenger_fare`, aunque el archivo tiene 24 columnas: es la poda de columnas. En la segunda, `PushedFilters` incluye `EqualTo(PULocationID,132)`: el filtro viaja hasta el lector Parquet, que puede saltarse grupos de filas cuyas estadísticas descartan el valor. Que un filtro se empuje no garantiza que se salte mucho: depende de cómo estén ordenados los datos dentro del archivo. Prueba cambiar el filtro por otra columna y vuelve a ejecutar la celda para ver cómo cambia `PushedFilters`.

## 4. Estrechas, anchas y el shuffle

Casi todo el costo de un trabajo de Spark depende de una sola pregunta: ¿para calcular esta fila, hace falta mirar otras filas? Si la respuesta es no, la operación es *estrecha* y cada partición trabaja sola. Si la respuesta es sí, la operación es *ancha* y hay que mover filas entre particiones, lo que se llama *shuffle*.

### 4.1 Transformaciones estrechas: cada partición trabaja sola

**El problema.** Cuando el trabajo sobre una fila no necesita ninguna otra fila, cada partición se puede procesar sola, sin enviar nada por la red. **La idea de ingeniería** es reconocer esas operaciones: `WHERE`, `SELECT` y `withColumn` son *estrechas* (cada partición de salida sale de una sola partición de entrada). Escalan bien, porque las particiones no se esperan entre sí: con más hilos se procesan más particiones a la vez. **Lo que muestra el plan**: ninguna línea `Exchange`, y el mismo número de particiones antes y después.

In [ ]:
# Celda 18
estrecha = base_v.where(F.col("trip_miles") > 0).select("PULocationID", "trip_miles")
print("Particiones antes :", base_v.rdd.getNumPartitions())
print("Particiones después:", estrecha.rdd.getNumPartitions())
print("\nLíneas del plan que importan:")
mostrar_plan(estrecha, ["Exchange", "Filter", "Project", "FileScan"])
print("\n¿Aparece Exchange (movimiento de filas entre particiones)?", "Exchange" in texto_plan(estrecha))

**Qué observar.** El número de particiones no cambió y el plan no tiene `Exchange`. Ese es el rasgo de una transformación estrecha.

### 4.2 Transformaciones anchas: el shuffle

**El problema.** Para contar viajes por zona hay que juntar todas las filas de la misma zona, y esas filas están repartidas en todas las particiones. **La idea de ingeniería** es el *shuffle*: cada partición reparte sus filas por clave (aplica una función hash a la zona) y las envía a la partición que le toca a esa clave. Eso implica escribir datos a disco, moverlos por la red (aquí, entre hilos) y leerlos de nuevo. Es la operación más cara de Spark, y es lo que hacen `GROUP BY`, `JOIN`, `ORDER BY` y `DISTINCT`. **Lo que muestra el plan**: una línea `Exchange` que parte el plan en dos etapas.

In [ ]:
# Celda 19
ancha = base_v.groupBy("PULocationID").agg(
    F.count(F.lit(1)).alias("viajes"), F.sum("base_passenger_fare").alias("ingreso"))

print("spark.sql.shuffle.partitions =", spark.conf.get("spark.sql.shuffle.partitions"),
      "(en cuántas particiones se reparte el resultado de un shuffle)")
print("\nPlan completo:")
mostrar_plan(ancha)
with cronometro("groupBy por zona (acción: collect)"):
    filas_zona = ancha.collect()
print("Filas del resultado:", len(filas_zona))

**Qué observar.** El plan se lee de abajo hacia arriba: `FileScan` lee, un `HashAggregate` agrega, un `Exchange hashpartitioning(PULocationID...)` reparte por zona y otro `HashAggregate` junta. Ahora abre la Spark UI, entra a *Jobs*, abre el último trabajo y mira *Stages*: verás **dos etapas**, y la frontera entre ellas es el `Exchange`. La primera etapa escribe datos de shuffle (*Shuffle Write*) y la segunda los lee (*Shuffle Read*). Fíjate también en que `spark.sql.shuffle.partitions` vale 200, pero la etapa que lee el shuffle no corre 200 tareas: la *ejecución adaptativa* (AQE, activa por defecto desde Spark 3.2) mira el tamaño real de los datos del shuffle y junta las particiones pequeñas en pocas, así que la Spark UI mostrará menos de 200 tareas en esa etapa. Antes de seguir, la Celda 20 consulta esos números de la Spark UI desde Python, para que no dependas del navegador.

### 4.3 GROUP BY sí se paraleliza: lo hace en dos fases

Puede parecer que agrupar obliga a mover los 19.6 millones de filas. No es así, y la razón es que Spark agrupa en **dos fases**. Primero, cada partición agrega *sus propias* filas (`HashAggregate` con funciones `partial_count`, `partial_sum`): de millones de filas queda una fila por zona y por partición. Solo esas filas parciales viajan por el shuffle. Segundo, después del `Exchange`, un segundo `HashAggregate` (el *final*) suma los parciales de cada zona. Por eso `COUNT`, `SUM`, `MIN` y `MAX` escalan bien, y también `AVG`: Spark lo parte en una suma y un conteo parciales y divide al final. No todas las agregaciones se pueden partir así: calcular una mediana exacta, por ejemplo, necesita ver todos los valores.

In [ ]:
# Celda 20
print("Fases de agregación en el plan:")
mostrar_plan(ancha, ["HashAggregate", "Exchange"])


def id_de_etapa(etapa):
    return etapa["stageId"]


def etapas_recientes(n=3):
    """Etapas ya terminadas, leídas de la API de la Spark UI (la UI vive mientras la sesión esté viva)."""
    import urllib.request
    sc = spark.sparkContext
    url = f"{sc.uiWebUrl}/api/v1/applications/{sc.applicationId}/stages?status=complete"
    try:
        with urllib.request.urlopen(url, timeout=10) as r:
            etapas = json.load(r)
    except Exception as e:
        print("No se pudo leer la Spark UI desde Python:", type(e).__name__, "- míralas en el navegador.")
        return
    etapas_ordenadas = sorted(etapas, key=id_de_etapa, reverse=True)
    for e in etapas_ordenadas[:n]:
        print(f"etapa {e['stageId']:>3} | tareas {e.get('numTasks', 0):>4} | "
              f"shuffle escrito: {fmt(e.get('shuffleWriteRecords', 0)):>12} filas ({e.get('shuffleWriteBytes', 0) / MB:,.1f} MB) | "
              f"shuffle leído: {fmt(e.get('shuffleReadRecords', 0)):>12} filas | {e.get('name', '')[:40]}")


print("\nÚltimas etapas ejecutadas (el groupBy fue la más reciente que escribió shuffle):")
etapas_recientes(4)

**Qué observar.** Primero, en el plan: un `HashAggregate` con `partial_count(1)` y `partial_sum(...)` antes del `Exchange`, y otro con `count(1)` y `sum(...)` después. Segundo, en la tabla de etapas: la etapa que escribió shuffle escribió **muy pocas filas comparadas con las 19 663 930 del archivo** (la celda imprime la cifra real). Esa diferencia es el ahorro de la agregación parcial: se movieron los resúmenes de cada partición, no los viajes. La cifra es, como máximo, el número de particiones de lectura por el número de zonas: en el Code Editor son 4 particiones con hasta 262 zonas cada una, y la medición dio 1 041 filas, porque no todas las zonas aparecen en todas las particiones. Tercero, la etapa que lee el shuffle trabaja con tantas tareas como `spark.sql.shuffle.partitions` o menos: la ejecución adaptativa (AQE) junta las particiones pequeñas.

### 4.4 Dos lugares donde Spark se queda trabajando como una sola máquina

Aunque tenga muchos hilos, hay operaciones que obligan a Spark a juntar todo en una sola tarea. Dos de ellas se pueden medir en este notebook. Hay otras dos, las claves desbalanceadas y los archivos pequeños, que se miden en la clase 7 cuando dimensionas el cómputo.

**a) Una ventana sin `PARTITION BY`.** Una ventana como `ROW_NUMBER() OVER (ORDER BY pickup_datetime)` necesita ordenar *todas* las filas juntas para numerarlas. Sin `PARTITION BY`, Spark no tiene por dónde partir y mueve todo a **una sola partición**, es decir, a una sola tarea. Con `PARTITION BY` zona, cada zona se numera por separado y las zonas se reparten entre las particiones. Para no agotar la memoria, la prueba usa una muestra (5 % de las filas) del archivo.

In [ ]:
# Celda 21
FRACCION_VENTANA = 0.05
muestra_v = base_v.select("PULocationID", "pickup_datetime", "base_passenger_fare").sample(False, FRACCION_VENTANA, seed=42)

sin_particion = muestra_v.withColumn("n", F.row_number().over(Window.orderBy("pickup_datetime")))
con_particion = muestra_v.withColumn("n", F.row_number().over(
    Window.partitionBy("PULocationID").orderBy("pickup_datetime")))

print("=== Ventana SIN PARTITION BY ===")
mostrar_plan(sin_particion, ["Exchange", "Window"])
print("\n=== Ventana CON PARTITION BY PULocationID ===")
mostrar_plan(con_particion, ["Exchange", "Window"])

with cronometro("ventana sin PARTITION BY"):
    maximo_1 = sin_particion.agg(F.max("n")).first()[0]
with cronometro("ventana con PARTITION BY zona"):
    maximo_2 = con_particion.agg(F.max("n")).first()[0]
print(f"\nFilas de la muestra: {fmt(muestra_v.count())} | mayor número de fila sin partición: {fmt(maximo_1)} | con partición: {fmt(maximo_2)}")

**Qué observar.** En el plan sin partición aparece `Exchange SinglePartition`: todas las filas viajan a una partición. Spark además imprime en el registro del proceso (no en el notebook) la advertencia `No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation`; es la forma en que Spark te avisa de este problema. En el plan con partición aparece `Exchange hashpartitioning(PULocationID...)`, que sí reparte. Lee los dos tiempos; el efecto crece con los datos, y con 19.6 millones de filas sin partición, una sola tarea de un solo ejecutor tendría que sostener y ordenar todas las filas. En un clúster, ese ejecutor es una sola máquina, sin importar cuántas haya; aquí, ese ejecutor vive dentro del proceso de 2 GB. La Celda 21 mide el efecto sobre una muestra.

**b) Traer todo al driver con `collect()` o `toPandas()`.** Estas dos acciones traen los datos al driver, que es un solo proceso con 2 GB de memoria. Está bien traer un resultado pequeño (el resumen de 262 zonas) y es una mala idea traer los datos crudos. La Celda 22 mide tres cosas: traer un millón de filas crudas de dos columnas; calcular desde cero el resumen de las 262 zonas sobre los 19.6 millones de viajes y traerlo; y traer ese mismo resumen cuando ya estaba calculado, que es el DataFrame `ancha` de la Celda 19 (Spark no repite su etapa de agregación porque conserva los archivos del shuffle). Así se separa cuánto de la cuenta es calcular y cuánto es traer. Al final compara la memoria con la estimación de enero completo con las 24 columnas (la sección 2.2).

In [ ]:
# Celda 22
with cronometro("toPandas: 1 millón de filas, 2 columnas"):
    un_millon = base_v.select("PULocationID", "base_passenger_fare").limit(1_000_000).toPandas()
mb_un_millon = un_millon.memory_usage(deep=True).sum() / MB

# El mismo resumen calculado desde cero: un DataFrame nuevo, sin los archivos de shuffle de `ancha`.
with cronometro("toPandas: resumen por zona calculado desde cero"):
    resumen_desde_cero = (base_v.groupBy("PULocationID")
                          .agg(F.count(F.lit(1)).alias("viajes"), F.sum("base_passenger_fare").alias("ingreso"))
                          .toPandas())

# `ancha` ya se ejecutó en la sección 4.2, así que aquí solo se trae el resultado.
with cronometro("toPandas: el mismo resumen, ya calculado (solo traerlo)"):
    resumen_zonas = ancha.toPandas()
mb_resumen = resumen_zonas.memory_usage(deep=True).sum() / MB

print(f"\n1 millón de filas, 2 columnas: {mb_un_millon:,.1f} MB en el proceso Python | resumen por zona ({len(resumen_zonas)} filas): {mb_resumen:,.3f} MB")
try:
    print(f"Estimación (sección 2.2) de enero completo con todas las columnas: {est_mes:,.0f} MB; el driver de Spark tiene {spark.sparkContext.getConf().get('spark.driver.memory')}.")
except NameError:
    print("Ejecuta la sección 2.2 para ver la estimación de enero completo.")

**Qué observar.** Hay tres tiempos y significan cosas distintas. Traer el resumen ya calculado cuesta casi nada (en nuestra medición del 16 nov. 2026, 0.08 segundos), pero ese tiempo es solo el de traerlo: la agregación ya estaba hecha. Calcularlo desde cero sobre los 19.6 millones de viajes cuesta lo que cuesta el `groupBy` de la sección 4.2 más traerlo (2.0 segundos en nuestra medición). Un millón de filas crudas de dos columnas tardó 8.6 segundos. Y traer un millón de filas crudas cuesta más que traer el resumen aunque no calcule nada, y su memoria crece con el número de filas. La regla de ingeniería es que **al driver se traen resultados, no datos**: agrega, filtra o toma una muestra en Spark, y solo entonces `toPandas()`.

Para cerrar este bloque, un resumen de las cuatro situaciones que has visto hasta aquí:

| Situación | Qué ves en el plan o en la Spark UI | Qué hacer |
|---|---|---|
| Operación estrecha (`where`, `select`, `withColumn`) | Ningún `Exchange`; las mismas particiones antes y después | Nada: escala sola |
| Operación ancha (`groupBy`, `join`, `orderBy`, `distinct`) | Un `Exchange` que parte el trabajo en dos etapas | Reducir antes de mover: filtrar y seleccionar primero; usar agregaciones parciales |
| Ventana sin `PARTITION BY` | `Exchange SinglePartition` y la advertencia de Spark | Dar siempre una partición a la ventana |
| `collect()` o `toPandas()` de datos crudos | Un solo proceso con memoria limitada | Traer resultados, no datos |

## 5. Broadcast: evitar el shuffle copiando la tabla pequeña

**El problema.** Para saber a qué zona pertenece cada viaje hay que hacer un `JOIN` entre los 19.6 millones de viajes y la tabla de zonas, que tiene 265 filas. Un `JOIN` normal es ancho: para unir por clave, Spark manda **las dos tablas** por un shuffle, y reparte por `LocationID` los 19.6 millones de viajes solo para encontrar el nombre de 265 zonas. **La idea de ingeniería** es notar que una de las dos tablas es diminuta: en lugar de mover los viajes, se copia la tabla de zonas completa a cada ejecutor (*broadcast*), y cada partición de viajes resuelve su `JOIN` localmente, sin mover ninguna fila de viajes. **Lo que resuelve** es eliminar el shuffle del lado grande. **Lo que no resuelve**: si la tabla «pequeña» no cabe en la memoria de cada ejecutor, el broadcast falla o satura la memoria. Por eso Spark solo lo aplica de forma automática cuando estima que la tabla mide menos de un umbral (`spark.sql.autoBroadcastJoinThreshold`, 10 MB por defecto). Esa estimación puede fallar, y por eso `enriquecer` en `silver.py` lo pide de forma explícita con `F.broadcast`. Pedirlo explícito tiene un costo: el umbral deja de protegerte, y si la tabla crece, el riesgo es tuyo. La tabla pequeña debe caber en la memoria del driver y en la de cada ejecutor.

La Celda 23 desactiva el broadcast automático para poder comparar de forma justa: primero un `JOIN` normal (Spark elige `SortMergeJoin`) y luego el mismo con `F.broadcast`. Usa una fracción de los viajes para que la prueba no tarde minutos; puedes subir `FRACCION_JOIN` a 1.0 si quieres usarlos todos. Al terminar, restaura la configuración original.

In [ ]:
# Celda 23
zonas_sp = spark.read.option("header", True).csv(str(RUTA_ZONAS))

FRACCION_JOIN = 0.25
umbral_original = spark.conf.get("spark.sql.autoBroadcastJoinThreshold")
viajes_join = base_v.select("PULocationID", "base_passenger_fare").sample(False, FRACCION_JOIN, seed=7)
zonas_join = zonas_sp.select(F.col("LocationID").cast("int").alias("PULocationID"), "Borough", "Zone")

try:
    spark.conf.set("spark.sql.autoBroadcastJoinThreshold", "-1")      # sin broadcast automático
    sin_bc = viajes_join.join(zonas_join, "PULocationID", "left")
    con_bc = viajes_join.join(F.broadcast(zonas_join), "PULocationID", "left")

    print("=== JOIN normal ===")
    mostrar_plan(sin_bc, ["Join", "Exchange", "Sort"])
    print("\n=== JOIN con F.broadcast(zonas) ===")
    mostrar_plan(con_bc, ["Join", "Exchange", "Sort"])

    with cronometro("join SIN broadcast + groupBy borough"):
        r1 = sin_bc.groupBy("Borough").agg(F.count(F.lit(1)).alias("viajes")).orderBy("Borough").collect()
    with cronometro("join CON broadcast + groupBy borough"):
        r2 = con_bc.groupBy("Borough").agg(F.count(F.lit(1)).alias("viajes")).orderBy("Borough").collect()
finally:
    spark.conf.set("spark.sql.autoBroadcastJoinThreshold", umbral_original)

filas_1 = []
for fila in r1:
    filas_1.append(tuple(fila))
filas_2 = []
for fila in r2:
    filas_2.append(tuple(fila))
print("\nResultados iguales:", filas_1 == filas_2)
print("Umbral de broadcast restaurado a:", spark.conf.get("spark.sql.autoBroadcastJoinThreshold"))

**Qué observar.** En el primer plan aparecen `SortMergeJoin`, dos `Exchange hashpartitioning(PULocationID...)` (uno por cada lado, es decir, los viajes también viajan) y dos `Sort`. En el segundo aparecen `BroadcastHashJoin` y un `BroadcastExchange` que mueve solo la tabla de zonas: los viajes no pasan por ningún shuffle. Compara los dos tiempos y confirma que los resultados son iguales: el broadcast cambia *cómo* se calcula, no *qué* se calcula. En la Spark UI, pestaña *SQL / DataFrame*, abre las dos consultas y verás los dos diagramas.

## 6. Lo que te llevas al lab

En este notebook viste que Spark reparte los datos en particiones y las procesa con tareas; que describir un cálculo y ejecutarlo son momentos distintos (y que `explain()` te deja leer el plan antes de pagar por él); que las operaciones estrechas escalan sin comunicación mientras que las anchas necesitan un *shuffle*; que `GROUP BY` se paraleliza en dos fases y por eso mueve resúmenes y no filas; que una tabla pequeña se puede copiar a cada ejecutor con `broadcast` para evitar el shuffle del lado grande; y que una ventana sin partición y un `collect` de datos crudos son los dos lugares donde todo se junta en una sola tarea o proceso. En el lab de silver y gold de la clase 6 usarás cada una de estas decisiones: el `LEFT JOIN` con las zonas va con `broadcast`, y quitar duplicados es un shuffle que verás en el plan.

Antes de abrir el lab, apaga Spark con la Celda 24 (solo cabe un Spark a la vez en esta máquina).

In [ ]:
# Celda 24
spark.stop()
print("Spark cerrado. Ya puedes abrir el siguiente notebook.")